# Introduction to RAG

#🔐 How to Get Your Gemini API Key
To use the Gemini 1.5 Flash model in this notebook, you’ll need a free API key from Google AI Studio. Follow the steps below to generate one:

Step-by-Step Instructions:

1. Visit Google AI Studio

  Open this link in your browser: https://aistudio.google.com/app/apikey

2. Sign in with Your Google Account

  Make sure you're signed in with a Google account (Gmail, Workspace, etc.).

3. Agree to Terms

  If it's your first time using AI Studio, you'll be asked to accept the terms and conditions.

4. Generate an API Key

  Click the “Create API Key” button.

5. A key will be displayed. It will look something like this:
AI...123xyz

6. Copy Your API Key

  Make sure to copy and store it somewhere safe.

7. You’ll paste this key into the notebook cell below.

In [10]:
!pip uninstall -y google-generativeai
!pip install google-genai


In [ ]:
from google.genai import Client
from google.colab import userdata

# 1. Connect the API key via the new Client
client = Client(api_key=userdata.get('GEMINI'))

# 2. Set the prompt
prompt = "Summarize the concept of reinforcement learning in AI."

# 3. Call the newly supported model (gemini-3.6-flash or gemini-3.7-flash)
response = client.models.generate_content(
    model='gemini-3.7-flash',  # You can also use gemini-3.7-flash here
    contents=prompt,
)

# 4. Print the result
print(response.text)



ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [11]:
import time
from google.genai import Client
from google.colab import userdata

client = Client(api_key=userdata.get('GEMINI'))
prompt = "Summarize the concept of reinforcement learning in AI."

# Retry loop: will try 3 times if a 503 error occurs
for attempt in range(3):
    try:
        response = client.models.generate_content(
            model='gemini-3.6-flash',
            contents=prompt,
        )
        print(response.text)
        break # Exit the loop if it succeeds
    except Exception as e:
        if "503" in str(e) and attempt < 2:
            print(f"Server is under load. Attempt {attempt + 1} failed. Waiting 5 seconds before retrying...")
            time.sleep(5)  # wait for 5 seconds
        else:
            print("Error:", e)


**Reinforcement Learning (RL)** is a type of machine learning where an AI agent learns to make decisions by **trial and error**, interacting with an environment to achieve a specific goal. 

Unlike traditional machine learning, which relies on being fed correct answers (supervised learning) or finding hidden patterns (unsupervised learning), RL relies on a **system of rewards and penalties**. 

---

### The Core Analogy
Think of training a dog: 
* If the dog performs a trick correctly, you give it a treat (**positive reward**). 
* If it misbehaves, you withhold the treat or say "no" (**negative reward/penalty**). 
* Over time, the dog figures out which actions lead to treats and adapts its behavior to get as many treats as possible.

---

### How It Works: The 5 Key Components

An RL system operates in a continuous feedback loop consisting of:

1. **Agent:** The AI decision-maker (e.g., a self-driving car software).
2. **Environment:** The world the agent interacts with (e.g., the road



Your task is to use the optimal vector store you created in the last lab to build RAG App on the Book Crime and Punishment.
You may run this on local .py files instead of Colab or Jupyter notebooks for your ease. Use Gemini as your LLM.

## Step 1:
- Implement a chat app, where single turn conversations take place. i.e. question asked is answered directly, history is not maintained. Use streamlit for a clean interface.

## Step 2:
- Add history into conversation by saving conversation in variables. History will be saved in memory here, not stored. Your app should now be able to answer questions in multi-turn conversations within single session.

## Step 3:
- Use supabase or any other database to store chats. This will be a relational database, with a Chat table, containing Roles:(varchar: 'ai', 'user'), Message (varchar). Your app should now be able to answer questions in multi-turn conversations even if session restarts.

<br> <br>

## Resources:
- Streamlit: https://docs.streamlit.io/develop/tutorials/chat-and-llm-apps/build-conversational-apps
- Langchain: https://medium.com/data-science/retrieval-augmented-generation-rag-from-theory-to-langchain-implementation-4e9bd5f6a4f2
<br> <br>
Feel free to search for additional resources

## Setup: install packages + connect Gemini key
This all runs inside Colab. Streamlit will be opened in the browser from Colab via `localtunnel`.

In [12]:
!pip install -q streamlit chromadb google-genai supabase
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [13]:
import os
from google.colab import userdata

# Same GEMINI secret you set earlier (Colab -> Secrets)
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI')
print('Gemini key loaded into environment.')

Gemini key loaded into environment.


> ⚠️ **If the runtime restarts, make sure to run this cell (Cell 8) again.**
> The `GEMINI_API_KEY` environment variable only lives in memory for this session — it gets deleted as soon as you restart, which causes an "API key not available" error later on.

## Download the book text (Crime and Punishment)
This downloads directly from Project Gutenberg, no manual upload needed.

In [14]:
import requests
import time

url = "https://www.gutenberg.org/cache/epub/2554/pg2554.txt"
mirror_url = "https://gutenberg.pglaf.org/2/5/5/2554/2554.txt"  # backup mirror if main site times out

response = None
for attempt in range(4):
    try:
        current_url = url if attempt < 2 else mirror_url
        response = requests.get(current_url, timeout=15)
        response.raise_for_status()
        break
    except requests.exceptions.RequestException as e:
        print(f"Attempt {attempt + 1} failed: {e}")
        if attempt < 3:
            time.sleep(5)

if response is None:
    raise RuntimeError("Could not download the book after multiple attempts. Try again in a few minutes.")

with open("crime_and_punishment.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print(f"Downloaded {len(response.text):,} characters.")


Downloaded 1,176,911 characters.


## Build the vector store (Chroma + Gemini embeddings)
This is the optimal vector store from the "last lab" — the assumption here is Chroma + `text-embedding-004`. If you used a different vector DB (FAISS, Pinecone, etc.) in the last lab, only this cell needs to change; everything else stays the same.

In [15]:
!pip install google-genai chromadb


In [16]:
from google.colab import userdata  # Make sure to add this import at the top

# NOTE: previous version of this cell had a bug -
# `api_key=userdata.get('GEMINI'),` created a tuple and did nothing.
# The actual client/API key setup happens in Cell 8 and inside rag_utils.py,
# so this cell just re-confirms the key is loaded.
gemini_key = userdata.get('GEMINI')
print("Gemini key loaded:", bool(gemini_key))


Gemini key loaded: True


In [22]:
!pip install sentence-transformers






In [23]:
import os
import chromadb
import time

CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "crime_and_punishment"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# 1. Text Chunking Function
def chunk_text(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Read the file
with open("crime_and_punishment.txt", "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Total chunks generated: {len(chunks)}")

# 2. Chroma DB Setup
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("Old collection has been deleted.")
except Exception:
    pass

# We are not passing any custom embedding model,
# ChromaDB will use its own default free local model.
collection = chroma_client.create_collection(name=COLLECTION_NAME)

ids = [f"chunk_{i}" for i in range(len(chunks))]

print("\nChromaDB is generating and saving embeddings using the local model...")

# All chunks are added directly at once, with no API rate limit
batch_size = 100
for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]

    collection.add(
        documents=batch_chunks,
        ids=batch_ids
    )
    print(f"Successfully added chunks {i} to {min(i+batch_size, len(chunks))}")

print("\n🚀 Vector store is ready with local embeddings at:", CHROMA_DIR)


Total chunks generated: 1444
Purani collection delete kar di gayi hai.

ChromaDB local model se embeddings generate aur save kar raha hai...


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 52.3MiB/s]


Successfully added chunks 0 to 100
Successfully added chunks 100 to 200
Successfully added chunks 200 to 300
Successfully added chunks 300 to 400
Successfully added chunks 400 to 500
Successfully added chunks 500 to 600
Successfully added chunks 600 to 700
Successfully added chunks 700 to 800
Successfully added chunks 800 to 900
Successfully added chunks 900 to 1000
Successfully added chunks 1000 to 1100
Successfully added chunks 1100 to 1200
Successfully added chunks 1200 to 1300
Successfully added chunks 1300 to 1400
Successfully added chunks 1400 to 1444

🚀 Vector store local embeddings ke sath taiyar hai at: chroma_db


## Shared RAG helper file
All three steps (1, 2, 3) use this same `rag_utils.py` — for retrieval, prompt building, and the Gemini call.

In [24]:
%%writefile rag_utils.py
import os
import chromadb
from chromadb.utils import embedding_functions
from google.genai import Client

CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "crime_and_punishment"
GEMINI_MODEL = "gemini-2.0-flash"
EMBEDDING_MODEL = "models/gemini-embedding-001"


def get_gemini_client():
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("GEMINI_API_KEY environment variable not set.")
    return Client(api_key=api_key)


def get_collection():
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    embed_fn = embedding_functions.GoogleGenerativeAiEmbeddingFunction(
        api_key=os.environ.get("GEMINI_API_KEY"),
        model_name=EMBEDDING_MODEL,
    )
    return client.get_collection(name=COLLECTION_NAME, embedding_function=embed_fn)


def retrieve_context(query, k=4):
    collection = get_collection()
    results = collection.query(query_texts=[query], n_results=k)
    return results["documents"][0]


def build_prompt(query, context_chunks, history=None):
    context_text = "\n\n---\n\n".join(context_chunks)

    history_text = ""
    if history:
        for turn in history:
            speaker = "User" if turn["role"] == "user" else "Assistant"
            history_text += f"{speaker}: {turn['message']}\n"

    prompt = f"""You are a helpful assistant answering questions about the novel "Crime and Punishment" by Fyodor Dostoevsky.
Use ONLY the context below to answer. If the answer isn't in the context, say you don't know.

Context:
{context_text}
"""
    if history_text:
        prompt += f"\nConversation so far:\n{history_text}"

    prompt += f"\nQuestion: {query}\nAnswer:"
    return prompt


def ask_gemini(prompt):
    client = get_gemini_client()
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    return response.text

Writing rag_utils.py


## Step 1: Single-turn chat app
Each question is answered on its own; no history is maintained.

In [25]:
%%writefile step1_single_turn.py
import streamlit as st
from rag_utils import retrieve_context, build_prompt, ask_gemini

st.set_page_config(page_title="Crime and Punishment RAG - Step 1", page_icon="📖")
st.title("📖 Ask Crime and Punishment (Single Turn)")
st.caption("Each question is answered independently. No memory between questions.")

query = st.chat_input("Ask something about the book...")

if query:
    with st.chat_message("user"):
        st.write(query)

    with st.chat_message("assistant"):
        with st.spinner("Searching the book and thinking..."):
            context_chunks = retrieve_context(query, k=4)
            prompt = build_prompt(query, context_chunks)
            answer = ask_gemini(prompt)
        st.write(answer)

        with st.expander("Show retrieved context"):
            for i, chunk in enumerate(context_chunks, 1):
                st.markdown(f"**Chunk {i}:**")
                st.write(chunk)

Writing step1_single_turn.py


Run Step 1 app: run the cell below, wait a bit, then click the link (`...loca.lt`) that gets printed. The password is the same IP printed in the cell output — paste it on that page and click "Submit".

In [26]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run step1_single_turn.py &>/content/logs1.txt &
import time; time.sleep(5)
!npx localtunnel --port 8501

8.228.240.140
⠙⠹⠸⠼⠴⠦your url is: https://hip-spiders-juggle.loca.lt
^C


Before running the next step, **stop/interrupt** the cell above (by stopping the runtime), otherwise Step 1's streamlit process will keep running on port 8501.

## Step 2: Multi-turn chat app (in-memory history)
Conversation history is saved in memory (`st.session_state`) for the session — it gets deleted when the app restarts.

In [27]:
%%writefile step2_multi_turn.py
import streamlit as st
from rag_utils import retrieve_context, build_prompt, ask_gemini

st.set_page_config(page_title="Crime and Punishment RAG - Step 2", page_icon="📖")
st.title("📖 Ask Crime and Punishment (Multi-turn, In Memory)")
st.caption("Conversation history is kept only for this browser session.")

if "history" not in st.session_state:
    st.session_state.history = []

for turn in st.session_state.history:
    role = "user" if turn["role"] == "user" else "assistant"
    with st.chat_message(role):
        st.write(turn["message"])

query = st.chat_input("Ask something about the book...")

if query:
    st.session_state.history.append({"role": "user", "message": query})
    with st.chat_message("user"):
        st.write(query)

    with st.chat_message("assistant"):
        with st.spinner("Searching the book and thinking..."):
            context_chunks = retrieve_context(query, k=4)
            prompt = build_prompt(query, context_chunks, history=st.session_state.history[:-1])
            answer = ask_gemini(prompt)
        st.write(answer)

    st.session_state.history.append({"role": "ai", "message": answer})

Writing step2_multi_turn.py


In [28]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run step2_multi_turn.py &>/content/logs2.txt &
import time; time.sleep(5)
!npx localtunnel --port 8501

8.228.240.140
⠙⠹⠸⠼⠴your url is: https://silver-rats-pick.loca.lt
^C


## Step 3: Multi-turn chat app with persistent storage (Supabase)
First, run this SQL once in your Supabase project:
```sql
create table chat (
    id bigint generated always as identity primary key,
    session_id text not null,
    role varchar not null,
    message varchar not null,
    created_at timestamp with time zone default now()
);
```
Then add your Supabase URL and key to Colab secrets under the names `SUPABASE_URL` and `SUPABASE_KEY`.

> ⚠️ **Before running this cell, check that:** both `SUPABASE_URL` and `SUPABASE_KEY` exist in Colab's 🔑 Secrets panel, the spelling is exact, and each one has its **"Notebook access" toggle turned ON**. If the toggle is off, `userdata.get()` will return an empty value and `create_client()` will crash (producing a confusing `close()` traceback).

In [48]:
%%writefile step3_persistent_db.py
import os
import uuid
import streamlit as st
from supabase import create_client
from rag_utils import retrieve_context, build_prompt, ask_gemini

# ⚠️ PASTE YOUR OWN DETAILS DIRECTLY HERE (without using secrets):
SUPABASE_URL = "https://zrmmspbcwcgziohbjsxx.supabase.co"  # Paste your actual URL here
SUPABASE_KEY = "sb_publishable_DHGyV92kWBOLxwfEdSYV9Q_dYpq3i91" # Paste the key from your screenshot here

# Client initialization
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

st.set_page_config(page_title="Crime and Punishment RAG - Step 3", page_icon="📖")
st.title("📖 Ask Crime and Punishment (Persistent History)")
st.caption("Conversation history is saved in Supabase and survives restarts.")

with st.sidebar:
    st.subheader("Session")
    resume_id = st.text_input("Resume session ID (optional)")
    if st.button("Start new session"):
        st.session_state.session_id = str(uuid.uuid4())
    if resume_id:
        st.session_state.session_id = resume_id

if "session_id" not in st.session_state:
    st.session_state.session_id = str(uuid.uuid4())

st.sidebar.write("Current session ID (copy this to resume later):")
st.sidebar.code(st.session_state.session_id)


def load_history(session_id):
    res = (
        supabase.table("chat")
        .select("role, message")
        .eq("session_id", session_id)
        .order("created_at")
        .execute()
    )
    return res.data


def save_message(session_id, role, message):
    supabase.table("chat").insert(
        {"session_id": session_id, "role": role, "message": message}
    ).execute()


history = load_history(st.session_state.session_id)

for turn in history:
    role = "user" if turn["role"] == "user" else "assistant"
    with st.chat_message(role):
        st.write(turn["message"])

query = st.chat_input("Ask something about the book...")

if query:
    save_message(st.session_state.session_id, "user", query)
    with st.chat_message("user"):
        st.write(query)

    with st.chat_message("assistant"):
        with st.spinner("Searching the book and thinking..."):
            context_chunks = retrieve_context(query, k=4)
            prompt = build_prompt(query, context_chunks, history=history)
            answer = ask_gemini(prompt)
        st.write(answer)

    save_message(st.session_state.session_id, "ai", answer)




Overwriting step3_persistent_db.py


In [49]:
!pkill streamlit


In [50]:
!wget -q -O - ://icanhazip.com
!streamlit run step3_persistent_db.py &>/content/logs3.txt &
import time; time.sleep(5)
!npx localtunnel --port 8501


⠙⠹⠸⠼⠴your url is: https://late-phones-cover.loca.lt
^C


# RAG Evaluation

## Step 4
Evaluate your RAG App using ragas:

- https://docs.ragas.io/en/stable/getstarted/evals/
- https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/
- https://docs.ragas.io/en/stable/howtos/applications/